#  ⭐ `dim_notificacoes`

In [63]:
%run ../../config/bootstrap.py

import pandas as pd
import numpy as np
from pathlib import Path
from utils import get_project_root, normalize_date_column, normalize_rows, apply_mappings, fuzzy_merge##, normalize_duration, expandir_gravidade_wide
project_root = get_project_root() 

In [64]:
project_root

WindowsPath('C:/Users/janet/Documents/VigiMed/vigimed')

# 🥉 Bronze 

Raw Data
as is production, low quality

In [65]:
path = Path(project_root) / "data/01_bronze/anvisa/Notificacoes/Notificacoes.parquet"
bronze = pd.read_parquet(path) 
bronze.columns

Index(['UF', 'TIPO_ENTRADA_VIGIMED', 'RECEBIDO_DE',
       'IDENTIFICACAO_NOTIFICACAO', 'DATA_INCLUSAO_SISTEMA',
       'DATA_ULTIMA_ATUALIZACAO', 'DATA_NOTIFICACAO', 'TIPO_NOTIFICACAO',
       'NOTIFICACAO_PARENT_CHILD', 'DATA_NASCIMENTO', 'IDADE_MOMENTO_REACAO',
       'GRUPO_IDADE', 'IDADE_GESTACIONAL_MOMENTO_REACAO', 'SEXO', 'GESTANTE',
       'LACTANTE', 'PESO_KG', 'ALTURA_CM', 'REACAO_EVENTO_ADVERSO_MEDDRA',
       'GRAVE', 'GRAVIDADE', 'DESFECHO', 'DATA_INICIO_HORA', 'DATA_FINAL_HORA',
       'DURACAO', 'RELACAO_MEDICAMENTO_EVENTO', 'NOME_MEDICAMENTO_WHODRUG',
       'ACAO_ADOTADA', 'NOTIFICADOR', 'ATUALIZADO'],
      dtype='object')

In [66]:
bronze.head(2)

,UF,TIPO_ENTRADA_VIGIMED,RECEBIDO_DE,IDENTIFICACAO_NOTIFICACAO,DATA_INCLUSAO_SISTEMA,DATA_ULTIMA_ATUALIZACAO,DATA_NOTIFICACAO,TIPO_NOTIFICACAO,NOTIFICACAO_PARENT_CHILD,DATA_NASCIMENTO,IDADE_MOMENTO_REACAO,GRUPO_IDADE,IDADE_GESTACIONAL_MOMENTO_REACAO,SEXO,GESTANTE,LACTANTE,PESO_KG,ALTURA_CM,REACAO_EVENTO_ADVERSO_MEDDRA,GRAVE,GRAVIDADE,DESFECHO,DATA_INICIO_HORA,DATA_FINAL_HORA,DURACAO,RELACAO_MEDICAMENTO_EVENTO,NOME_MEDICAMENTO_WHODRUG,ACAO_ADOTADA,NOTIFICADOR,ATUALIZADO
0,SP,Empresas Farmacęuticas,Empresa Farmacęutica,BR-ANVISA-300212656,20230928,20230928,None,Notificaçăo espontânea,None,19900131,30 ano,None,None,Feminino,Năo,Năo,68.0,165,Hemiparesia,Sim,Outro efeito clinicamente significativo,Recuperado,20210125,20210206,12 dia,Concomitante,Tamisa,Năo aplicável,Médico,2026-01-15
1,SP,Empresas Farmacęuticas,Empresa Farmacęutica,BR-ANVISA-300208322,20230901,20230901,None,Notificaçăo espontânea,None,None,None,None,None,Feminino,Năo,Năo,None,None,Cefaleia,Năo,Outro efeito clinicamente significativo,Desconhecido,20210122,None,None,Suspeito,CoronaVac,Năo aplicável,Consumidor ou outro năo profissional de saúde,2026-01-15


# 🥈 Silver

normalized data, medium quality

In [67]:
silver = bronze.copy()
silver.shape

(311134, 30)

## ✅ hash

In [68]:
from utils import build_row_hash

silver["HASH_BRONZE"] = build_row_hash(
    silver, 
    silver.columns.difference(['ATUALIZADO']).tolist(), 
    algo="sha256", 
    sep="|"
)

## ✅ Missing

In [69]:
silver.isna().sum()

UF                                   81507
TIPO_ENTRADA_VIGIMED                    70
RECEBIDO_DE                          55666
IDENTIFICACAO_NOTIFICACAO                0
DATA_INCLUSAO_SISTEMA                    0
DATA_ULTIMA_ATUALIZACAO                 70
DATA_NOTIFICACAO                    151881
TIPO_NOTIFICACAO                         0
NOTIFICACAO_PARENT_CHILD            310006
DATA_NASCIMENTO                      80710
IDADE_MOMENTO_REACAO                110538
GRUPO_IDADE                         183974
IDADE_GESTACIONAL_MOMENTO_REACAO    310802
SEXO                                 11687
GESTANTE                            160811
LACTANTE                            160814
PESO_KG                             189266
ALTURA_CM                           244428
REACAO_EVENTO_ADVERSO_MEDDRA           317
GRAVE                                69292
GRAVIDADE                           168932
DESFECHO                             49933
DATA_INICIO_HORA                     85453
DATA_FINAL_

In [70]:
hist_silver = silver.copy()
for col in silver.select_dtypes(include=['object']).columns:
    silver[col] = normalize_rows(silver[col])

In [71]:
hist_silver.isna().sum()

UF                                   81507
TIPO_ENTRADA_VIGIMED                    70
RECEBIDO_DE                          55666
IDENTIFICACAO_NOTIFICACAO                0
DATA_INCLUSAO_SISTEMA                    0
DATA_ULTIMA_ATUALIZACAO                 70
DATA_NOTIFICACAO                    151881
TIPO_NOTIFICACAO                         0
NOTIFICACAO_PARENT_CHILD            310006
DATA_NASCIMENTO                      80710
IDADE_MOMENTO_REACAO                110538
GRUPO_IDADE                         183974
IDADE_GESTACIONAL_MOMENTO_REACAO    310802
SEXO                                 11687
GESTANTE                            160811
LACTANTE                            160814
PESO_KG                             189266
ALTURA_CM                           244428
REACAO_EVENTO_ADVERSO_MEDDRA           317
GRAVE                                69292
GRAVIDADE                           168932
DESFECHO                             49933
DATA_INICIO_HORA                     85453
DATA_FINAL_

In [72]:
hist_silver.shape

(311134, 31)

## ✅ DATAS

In [73]:
colunas_data = ["DATA_INCLUSAO_SISTEMA", "DATA_ULTIMA_ATUALIZACAO", "DATA_NOTIFICACAO", "DATA_NASCIMENTO", "DATA_INICIO_HORA", "DATA_FINAL_HORA"]

In [74]:
hist_silver[colunas_data].info()
hist_silver[colunas_data].head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 311134 entries, 0 to 311133
Data columns (total 6 columns):
 #   Column                   Non-Null Count   Dtype 
---  ------                   --------------   ----- 
 0   DATA_INCLUSAO_SISTEMA    311134 non-null  object
 1   DATA_ULTIMA_ATUALIZACAO  311064 non-null  object
 2   DATA_NOTIFICACAO         159253 non-null  object
 3   DATA_NASCIMENTO          230424 non-null  object
 4   DATA_INICIO_HORA         225681 non-null  object
 5   DATA_FINAL_HORA          157812 non-null  object
dtypes: object(6)
memory usage: 14.2+ MB


,DATA_INCLUSAO_SISTEMA,DATA_ULTIMA_ATUALIZACAO,DATA_NOTIFICACAO,DATA_NASCIMENTO,DATA_INICIO_HORA,DATA_FINAL_HORA
0,20230928,20230928,None,19900131,20210125,20210206
1,20230901,20230901,None,None,20210122,None
2,20231006,20231006,None,19710522,20210203,None
3,20250721,20250721,None,19741123,20210302,None
4,20230927,20230927,None,None,None,None


In [75]:
for col in colunas_data:
    if col in hist_silver.columns:
        hist_silver[col] = normalize_date_column(hist_silver[col])

In [76]:
hist_silver[colunas_data].info()
hist_silver[colunas_data].head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 311134 entries, 0 to 311133
Data columns (total 6 columns):
 #   Column                   Non-Null Count   Dtype         
---  ------                   --------------   -----         
 0   DATA_INCLUSAO_SISTEMA    311134 non-null  datetime64[ns]
 1   DATA_ULTIMA_ATUALIZACAO  311064 non-null  datetime64[ns]
 2   DATA_NOTIFICACAO         158960 non-null  datetime64[ns]
 3   DATA_NASCIMENTO          229298 non-null  datetime64[ns]
 4   DATA_INICIO_HORA         169043 non-null  datetime64[ns]
 5   DATA_FINAL_HORA          107847 non-null  datetime64[ns]
dtypes: datetime64[ns](6)
memory usage: 14.2 MB


,DATA_INCLUSAO_SISTEMA,DATA_ULTIMA_ATUALIZACAO,DATA_NOTIFICACAO,DATA_NASCIMENTO,DATA_INICIO_HORA,DATA_FINAL_HORA
0,2023-09-28,2023-09-28,NaT,1990-01-31,2021-01-25,2021-02-06
1,2023-09-01,2023-09-01,NaT,NaT,2021-01-22,NaT
2,2023-10-06,2023-10-06,NaT,1971-05-22,2021-02-03,NaT
3,2025-07-21,2025-07-21,NaT,1974-11-23,2021-03-02,NaT
4,2023-09-27,2023-09-27,NaT,NaT,NaT,NaT


In [77]:
hist_silver.shape

(311134, 31)

## ✅ Mappings

- UF
- TIPO_ENTRADA_VIGIMED
- RECEBIDO_DE
- TIPO_NOTIFICACAO
- NOTIFICACAO_PARENT_CHILD
- GRUPO_IDADE
- SEXO
- GESTANTE
- LACTANTE
- GRAVE
- GRAVIDADE
- DESFECHO
- RELACAO_MEDICAMENTO_EVENTO
- ACAO_ADOTADA
- NOTIFICADOR

In [78]:
import unicodedata

def normalizar_texto(s):
    if pd.isna(s):
        return s
    s = str(s).strip().upper()
    s = unicodedata.normalize("NFKD", s)
    s = "".join(c for c in s if not unicodedata.combining(c))
    return s

hist_silver["UF"] = hist_silver["UF"].apply(normalizar_texto)
hist_silver["TIPO_ENTRADA_VIGIMED"] = hist_silver["TIPO_ENTRADA_VIGIMED"].apply(normalizar_texto)
hist_silver["RECEBIDO_DE"] = hist_silver["RECEBIDO_DE"].apply(normalizar_texto)
hist_silver["TIPO_NOTIFICACAO"] = hist_silver["TIPO_NOTIFICACAO"].apply(normalizar_texto)
hist_silver["GRAVIDADE"] = hist_silver["GRAVIDADE"].apply(normalizar_texto)
hist_silver["DESFECHO"] = hist_silver["DESFECHO"].apply(normalizar_texto)
hist_silver["RELACAO_MEDICAMENTO_EVENTO"] = hist_silver["RELACAO_MEDICAMENTO_EVENTO"].apply(normalizar_texto)
hist_silver["ACAO_ADOTADA"] = hist_silver["ACAO_ADOTADA"].apply(normalizar_texto)
hist_silver["NOTIFICADOR"] = hist_silver["NOTIFICADOR"].apply(normalizar_texto)

In [79]:
hist_silver["UF"] = (
    hist_silver["UF"]
    .astype(str)
    .str.strip()
    .str.upper()
)

In [80]:
# UF
uf_map = {
    "AC": 1, "AL": 2, "AP": 3, "AM": 4, "BA": 5, "CE": 6, "DF": 7, "ES": 8, "GO": 9, "MA": 10, 
    "MT": 11, "MS": 12, "MG": 13, "PA": 14, "PB": 15, "PR": 16, "PE": 17, "PI": 18, "RJ": 19,
    "RN": 20, "RS": 21, "RO": 22, "RR": 23, "SC": 24, "SP": 25, "SE": 26, "TO": 27
}

ufs_validas = set(uf_map.keys())
hist_silver.loc[~hist_silver["UF"].isin(ufs_validas), "UF"] = np.nan
hist_silver["UF_CHAVE"] = hist_silver["UF"].map(uf_map).fillna(0).astype("int64")
hist_silver["UF_VALOR"] = hist_silver["UF"].fillna("DESCONHECIDO")

# TIPO_ENTRADA_VIGIMED
tipo_entrada_vigimed_map = {
    "SERVICOS DE SAUDE": 1, "EMPRESAS FARMACEUTICAS": 2, "PACIENTES E PROFISSIONAIS DE SAUDE": 3,
    "SERVICOS DE VACINACAO": 4, "EREPORTING - INDUSTRIA, ENTRADA MANUAL DE DADOS": 5,
    "EREPORTING - INDUSTRIA, CARGA E2B": 6, "VIGIMOBILE": 7, "VIGIFLOW EFORMS": 8
}

# RECEBIDO_DE
recebido_de_map = {
    "CARGA E2B": 1, "ENTRADA MANUAL DE DADOS": 2, "AUTORIDADE REGULADORA": 3,
    "CENTRO REGIONAL DE FARMACOVIGILANCIA": 4, "EMPRESA FARMACEUTICA": 5,
    "OUTRO (P.EX. DISTRIBUIDORA, FINANCIADOR DE ESTUDO, ORGANIZACAO REPRESENTATIVA PARA PESQUISA CLINICA, OU ORGANIZACAO NAO-COMERCIAL)": 6,
    "PACIENTE/CONSUMIDOR": 7, "PROFISSIONAL DE SAUDE": 8
}

# TIPO_NOTIFICACAO
tipo_notificacao_map = {
    "NOTIFICACAO ESPONTANEA": 1, "NOTIFICACAO DE ESTUDO": 2, "OUTRO": 3
}

# NOTIFICACAO_PARENT_CHILD
notificacao_parent_child_map = {
    "SIM": 1, "NAO": 2
}

# GRUPO_IDADE
grupo_idade_map = {
    "FETO": 1, "NEONATO": 2, "INFANTIL": 3, "CRIANCA": 4, "ADOLESCENTE": 5,
    "ADULTO": 6, "IDOSO": 7, "OUTRO": 8
}

# SEXO
sexo_map = {
    "FEMININO": 1, "MASCULINO": 2
}

# GESTANTE
gestante_map = {
    "SIM": 1, "NAO": 2
}

# LACTANTE
lactante_map = {
    "SIM": 1, "NAO": 2
}

# GRAVE
grave_map = {
    "SIM": 1, "NAO": 2
}

gravidade_correcoes = { "AMEACA R VIDA": "AMEACA A VIDA" }
hist_silver["GRAVIDADE"] = hist_silver["GRAVIDADE"].apply(normalizar_texto)
hist_silver["GRAVIDADE"] = hist_silver["GRAVIDADE"].replace(gravidade_correcoes)

# GRAVIDADE
gravidade_map = {
    "OUTRO EFEITO CLINICAMENTE SIGNIFICATIVO": 1, "HOSPITALIZACAO": 2, "AMEACA A VIDA": 3,
    "RESULTOU EM OBITO": 4, "INCAPACIDADE PERSISTENTE OU SIGNIFICATIVA": 5,
    "ANOMALIA CONGENITA OU MALFORMACAO AO NASCER": 6
}

hist_silver["GRAVIDADE_CHAVE"] = (
    hist_silver["GRAVIDADE"]
    .map(gravidade_map)
    .fillna(0)
    .astype("int64")
)
inv_gravidade_map = {v: k for k, v in gravidade_map.items()}
hist_silver["GRAVIDADE_VALOR"] = hist_silver["GRAVIDADE_CHAVE"].map(inv_gravidade_map).fillna("DESCONHECIDO")

# DESFECHO
desfecho_map = {
    "RECUPERADO": 1, "EM RECUPERACAO": 2, "NAO RECUPERADO": 3, "FATAL": 4
}

# RELACAO_MEDICAMENTO_EVENTO
relacao_medicamento_evento_map = {
    "SUSPEITO": 1, "CONCOMITANTE": 2, "INTERACAO": 3, "MEDICAMENTO NAO ADMINISTRADO": 4
}

# ACAO_ADOTADA
acao_adotada_map = {
    "AUMENTO DA DOSE": 1, "NAO APLICAVEL": 2, "REDUCAO DA DOSE": 3, 
    "RETIRADA DO MEDICAMENTO": 4, "SEM ALTERACAO DA DOSE": 5
}

# NOTIFICADOR
notificador_map = {
    "ADVOGADO": 1, "CONSUMIDOR OU OUTRO NAO PROFISSIONAL DE SAUDE": 2,
    "FARMACEUTICO": 3, "MEDICO": 4, "OUTRO PROFISSIONAL DE SAUDE": 5
}

mappings_config = [
    {
        "kind": "dict",
        "col": "UF",
        "map": uf_map,
        "fillna_valor": "DESCONHECIDO",
        "fillna_chave": 0,
        "tipo_int64": True,
        "drop_original": False,
    },
    {
        "kind": "dict",
        "col": "TIPO_ENTRADA_VIGIMED",
        "map": tipo_entrada_vigimed_map,
        "fillna_valor": "DESCONHECIDO",
        "fillna_chave": 0,
        "tipo_int64": True,
        "drop_original": False,
    },
    {
        "kind": "dict",
        "col": "RECEBIDO_DE",
        "map": recebido_de_map,
        "fillna_valor": "DESCONHECIDO",
        "fillna_chave": 0,
        "tipo_int64": True,
        "drop_original": False,
    },
    {
        "kind": "dict",
        "col": "TIPO_NOTIFICACAO",
        "map": tipo_notificacao_map,
        "fillna_valor": "DESCONHECIDO",
        "fillna_chave": 0,
        "tipo_int64": True,
        "drop_original": False,
    },
    {
        "kind": "dict",
        "col": "NOTIFICACAO_PARENT_CHILD",
        "map": notificacao_parent_child_map,
        "fillna_valor": "DESCONHECIDO",
        "fillna_chave": 0,
        "tipo_int64": True,
        "drop_original": False,
    },
    {
        "kind": "dict",
        "col": "GRUPO_IDADE",
        "map": grupo_idade_map,
        "fillna_valor": "DESCONHECIDO",
        "fillna_chave": 0,
        "tipo_int64": True,
        "drop_original": False,
    },
    {
        "kind": "dict",
        "col": "SEXO",
        "map": sexo_map,
        "fillna_valor": "DESCONHECIDO",
        "fillna_chave": 0,
        "tipo_int64": True,
        "drop_original": False,
    },
    {
        "kind": "dict",
        "col": "GESTANTE",
        "map": gestante_map,
        "fillna_valor": "DESCONHECIDO",
        "fillna_chave": 0,
        "tipo_int64": True,
        "drop_original": False,
    },
    {
        "kind": "dict",
        "col": "LACTANTE",
        "map": lactante_map,
        "fillna_valor": "DESCONHECIDO",
        "fillna_chave": 0,
        "tipo_int64": True,
        "drop_original": False,
    },
    {
        "kind": "dict",
        "col": "GRAVE",
        "map": grave_map,
        "fillna_valor": "DESCONHECIDO",
        "fillna_chave": 0,
        "tipo_int64": True,
        "drop_original": False,
    },
    {
        "kind": "dict",
        "col": "GRAVIDADE",
        "map": gravidade_map,
        "fillna_valor": "DESCONHECIDO",
        "fillna_chave": 0,
        "tipo_int64": True,
        "drop_original": False,
    },
    {
        "kind": "dict",
        "col": "DESFECHO",
        "map": desfecho_map,
        "fillna_valor": "DESCONHECIDO",
        "fillna_chave": 0,
        "tipo_int64": True,
        "drop_original": False,
    },
    {
        "kind": "dict",
        "col": "RELACAO_MEDICAMENTO_EVENTO",
        "map": relacao_medicamento_evento_map,
        "fillna_valor": "DESCONHECIDO",
        "fillna_chave": 0,
        "tipo_int64": True,
        "drop_original": False,
    },
    {
        "kind": "dict",
        "col": "ACAO_ADOTADA",
        "map": acao_adotada_map,
        "fillna_valor": "DESCONHECIDO",
        "fillna_chave": 0,
        "tipo_int64": True,
        "drop_original": False,
    },
    {
        "kind": "dict",
        "col": "NOTIFICADOR",
        "map": notificador_map,
        "fillna_valor": "DESCONHECIDO",
        "fillna_chave": 0,
        "tipo_int64": True,
        "drop_original": False,
    },
]

In [81]:
hist_silver = apply_mappings(hist_silver, mappings_config)

In [82]:
hist_silver.shape

(311134, 61)

In [83]:
hist_silver[['UF','UF_VALOR','UF_CHAVE']].value_counts(dropna=False)

UF   UF_VALOR      UF_CHAVE
SP   SP            25          117317
NaN  DESCONHECIDO  0            81755
MG   MG            13           20254
BA   BA            5            11452
RJ   RJ            19           10871
DF   DF            7            10717
SC   SC            24            9291
RS   RS            21            8592
CE   CE            6             7915
PR   PR            16            7564
GO   GO            9             5921
ES   ES            8             5518
RN   RN            20            3691
PI   PI            18            1697
MA   MA            10            1623
PA   PA            14            1439
PE   PE            17            1398
AM   AM            4             1366
PB   PB            15             870
MS   MS            12             581
MT   MT            11             398
SE   SE            26             396
AL   AL            2              316
RO   RO            22             101
TO   TO            27              53
AP   AP            3  

In [84]:
hist_silver[['TIPO_ENTRADA_VIGIMED', 'TIPO_ENTRADA_VIGIMED_VALOR', 'TIPO_ENTRADA_VIGIMED_CHAVE']].value_counts(dropna=False)

TIPO_ENTRADA_VIGIMED                             TIPO_ENTRADA_VIGIMED_VALOR                       TIPO_ENTRADA_VIGIMED_CHAVE
SERVICOS DE SAUDE                                SERVICOS DE SAUDE                                1                             152367
EMPRESAS FARMACEUTICAS                           EMPRESAS FARMACEUTICAS                           2                              79444
PACIENTES E PROFISSIONAIS DE SAUDE               PACIENTES E PROFISSIONAIS DE SAUDE               3                              37774
EREPORTING - INDUSTRIA, ENTRADA MANUAL DE DADOS  EREPORTING - INDUSTRIA, ENTRADA MANUAL DE DADOS  5                              11710
EREPORTING - INDUSTRIA, CARGA E2B                EREPORTING - INDUSTRIA, CARGA E2B                6                               9996
SERVICOS DE VACINACAO                            SERVICOS DE VACINACAO                            4                               9321
VIGIMOBILE                                       VIGIMOBILE      

In [85]:
hist_silver[['RECEBIDO_DE', 'RECEBIDO_DE_VALOR', 'RECEBIDO_DE_CHAVE']].value_counts(dropna=False)

RECEBIDO_DE                                                                                                                         RECEBIDO_DE_VALOR                                                                                                                   RECEBIDO_DE_CHAVE
PROFISSIONAL DE SAUDE                                                                                                               PROFISSIONAL DE SAUDE                                                                                                               8                    104704
EMPRESA FARMACEUTICA                                                                                                                EMPRESA FARMACEUTICA                                                                                                                5                    101702
NaN                                                                                                                                 DE

In [86]:
hist_silver[['TIPO_NOTIFICACAO', 'TIPO_NOTIFICACAO_VALOR', 'TIPO_NOTIFICACAO_CHAVE']].value_counts(dropna=False)

TIPO_NOTIFICACAO                                TIPO_NOTIFICACAO_VALOR                          TIPO_NOTIFICACAO_CHAVE
NOTIFICACAO ESPONTANEA                          NOTIFICACAO ESPONTANEA                          1                         268975
NOTIFICACAO DE ESTUDO                           NOTIFICACAO DE ESTUDO                           2                          22119
OUTRO                                           OUTRO                                           3                          19855
NAO DISPONIVEL PELO NOTIFICADOR (DESCONHECIDO)  NAO DISPONIVEL PELO NOTIFICADOR (DESCONHECIDO)  0                            185
Name: count, dtype: int64

In [87]:
hist_silver[['NOTIFICACAO_PARENT_CHILD', 'NOTIFICACAO_PARENT_CHILD_VALOR', 'NOTIFICACAO_PARENT_CHILD_CHAVE']].value_counts(dropna=False)

NOTIFICACAO_PARENT_CHILD  NOTIFICACAO_PARENT_CHILD_VALOR  NOTIFICACAO_PARENT_CHILD_CHAVE
NaN                       DESCONHECIDO                    0                                 310006
Sim                       SIM                             1                                   1128
Name: count, dtype: int64

In [88]:
hist_silver[['GRUPO_IDADE', 'GRUPO_IDADE_VALOR', 'GRUPO_IDADE_CHAVE']].value_counts(dropna=False)

GRUPO_IDADE  GRUPO_IDADE_VALOR  GRUPO_IDADE_CHAVE
NaN          DESCONHECIDO       0                    183974
Adulto       ADULTO             6                     71598
Idoso        IDOSO              7                     42827
Criança      CRIANÇA            4                      4389
Infantil     INFANTIL           3                      3734
Adolescente  ADOLESCENTE        5                      3139
Neonato      NEONATO            2                      1409
Feto         FETO               1                        64
Name: count, dtype: int64

In [89]:
hist_silver[['SEXO', 'SEXO_VALOR', 'SEXO_CHAVE']].value_counts(dropna=False)

SEXO          SEXO_VALOR    SEXO_CHAVE
Feminino      FEMININO      1             186527
Masculino     MASCULINO     2             111891
NaN           DESCONHECIDO  0              11687
Desconhecido  DESCONHECIDO  0               1029
Name: count, dtype: int64

In [90]:
hist_silver[['GESTANTE', 'GESTANTE_VALOR', 'GESTANTE_CHAVE']].value_counts(dropna=False)

GESTANTE  GESTANTE_VALOR  GESTANTE_CHAVE
NaN       DESCONHECIDO    0                 160811
Năo       NĂO             2                 148633
Sim       SIM             1                   1690
Name: count, dtype: int64

In [91]:
hist_silver[['LACTANTE', 'LACTANTE_VALOR', 'LACTANTE_CHAVE']].value_counts(dropna=False)

LACTANTE  LACTANTE_VALOR  LACTANTE_CHAVE
NaN       DESCONHECIDO    0                 160814
Năo       NĂO             2                 148497
Sim       SIM             1                   1823
Name: count, dtype: int64

In [92]:
hist_silver[['GRAVE', 'GRAVE_VALOR', 'GRAVE_CHAVE']].value_counts(dropna=False)

GRAVE  GRAVE_VALOR   GRAVE_CHAVE
Sim    SIM           1              136768
Năo    NĂO           2              105074
NaN    DESCONHECIDO  0               69292
Name: count, dtype: int64

In [93]:
hist_silver[['GRAVIDADE', 'GRAVIDADE_VALOR', 'GRAVIDADE_CHAVE']].value_counts(dropna=False)

GRAVIDADE                                    GRAVIDADE_VALOR                              GRAVIDADE_CHAVE
NaN                                          DESCONHECIDO                                 0                  168932
OUTRO EFEITO CLINICAMENTE SIGNIFICATIVO      OUTRO EFEITO CLINICAMENTE SIGNIFICATIVO      1                   82979
HOSPITALIZACAO                               HOSPITALIZACAO                               2                   31898
RESULTOU EM OBITO                            RESULTOU EM OBITO                            4                   11049
AMEACA A VIDA                                AMEACA A VIDA                                3                    9952
INCAPACIDADE PERSISTENTE OU SIGNIFICATIVA    INCAPACIDADE PERSISTENTE OU SIGNIFICATIVA    5                    6179
ANOMALIA CONGENITA OU MALFORMACAO AO NASCER  ANOMALIA CONGENITA OU MALFORMACAO AO NASCER  6                     145
Name: count, dtype: int64

In [94]:
hist_silver[['DESFECHO', 'DESFECHO_VALOR', 'DESFECHO_CHAVE']].value_counts(dropna=False)

DESFECHO        DESFECHO_VALOR  DESFECHO_CHAVE
RECUPERADO      RECUPERADO      1                 146660
DESCONHECIDO    DESCONHECIDO    0                  53293
NaN             DESCONHECIDO    0                  49933
EM RECUPERACAO  EM RECUPERACAO  2                  28389
NAO RECUPERADO  NAO RECUPERADO  3                  21947
FATAL           FATAL           4                  10912
Name: count, dtype: int64

In [95]:
hist_silver[['RELACAO_MEDICAMENTO_EVENTO', 'RELACAO_MEDICAMENTO_EVENTO_VALOR', 'RELACAO_MEDICAMENTO_EVENTO_CHAVE']].value_counts(dropna=False)

RELACAO_MEDICAMENTO_EVENTO    RELACAO_MEDICAMENTO_EVENTO_VALOR  RELACAO_MEDICAMENTO_EVENTO_CHAVE
SUSPEITO                      SUSPEITO                          1                                   300007
MEDICAMENTO NAO ADMINISTRADO  MEDICAMENTO NAO ADMINISTRADO      4                                     8418
CONCOMITANTE                  CONCOMITANTE                      2                                     1944
INTERACAO                     INTERACAO                         3                                      764
NaN                           DESCONHECIDO                      0                                        1
Name: count, dtype: int64

In [96]:
hist_silver[['ACAO_ADOTADA', 'ACAO_ADOTADA_VALOR', 'ACAO_ADOTADA_CHAVE']].value_counts(dropna=False)

ACAO_ADOTADA             ACAO_ADOTADA_VALOR       ACAO_ADOTADA_CHAVE
NaN                      DESCONHECIDO             0                     126091
RETIRADA DO MEDICAMENTO  RETIRADA DO MEDICAMENTO  4                      71728
NAO APLICAVEL            NAO APLICAVEL            2                      37552
DESCONHECIDO             DESCONHECIDO             0                      35680
SEM ALTERACAO DA DOSE    SEM ALTERACAO DA DOSE    5                      31763
REDUCAO DA DOSE          REDUCAO DA DOSE          3                       5609
AUMENTO DA DOSE          AUMENTO DA DOSE          1                       2711
Name: count, dtype: int64

In [97]:
hist_silver[['NOTIFICADOR', 'NOTIFICADOR_VALOR', 'NOTIFICADOR_CHAVE']].value_counts(dropna=False)

NOTIFICADOR                                    NOTIFICADOR_VALOR                              NOTIFICADOR_CHAVE
FARMACEUTICO                                   FARMACEUTICO                                   3                    121082
CONSUMIDOR OU OUTRO NAO PROFISSIONAL DE SAUDE  CONSUMIDOR OU OUTRO NAO PROFISSIONAL DE SAUDE  2                     85054
OUTRO PROFISSIONAL DE SAUDE                    OUTRO PROFISSIONAL DE SAUDE                    5                     65862
MEDICO                                         MEDICO                                         4                     20474
NaN                                            DESCONHECIDO                                   0                     18164
ADVOGADO                                       ADVOGADO                                       1                       498
Name: count, dtype: int64

## ✅ IDADE_MOMENTO_REACAO

In [98]:
from utils.not_normalize_idade_momento_reacao import normalize_idade_momento_reacao

In [99]:
hist_silver = normalize_idade_momento_reacao(hist_silver, coluna="IDADE_MOMENTO_REACAO")

In [100]:
hist_silver[['IDADE_MOMENTO_REACAO','IDADE_MOMENTO_REACAO_TIPO_CHAVE','IDADE_MOMENTO_REACAO_TIPO_VALOR','IDADE_MOMENTO_REACAO_VALOR']].value_counts(dropna=False).head(10)

IDADE_MOMENTO_REACAO  IDADE_MOMENTO_REACAO_TIPO_CHAVE  IDADE_MOMENTO_REACAO_TIPO_VALOR  IDADE_MOMENTO_REACAO_VALOR
NaN                   0                                DESCONHECIDO                     NaN                           110538
60 ano                1                                ANO                              60.0                            3481
40 ano                1                                ANO                              40.0                            3312
42 ano                1                                ANO                              42.0                            3281
65 ano                1                                ANO                              65.0                            3279
61 ano                1                                ANO                              61.0                            3276
59 ano                1                                ANO                              59.0                            3275
63 ano    

In [101]:
hist_silver.shape

(311134, 64)

## ✅ DURACAO

In [102]:
from utils.not_normalize_duracao import normalize_duracao

In [103]:
hist_silver = normalize_duracao(hist_silver, coluna="DURACAO")

In [104]:
hist_silver[['DURACAO','DURACAO_TIPO_CHAVE','DURACAO_TIPO_VALOR','DURACAO_VALOR']].value_counts(dropna=False).head(10)

DURACAO    DURACAO_TIPO_CHAVE  DURACAO_TIPO_VALOR  DURACAO_VALOR
NaN        0                   DESCONHECIDO        NaN              204258
           0                   DESCONHECIDO        NaN               31565
1 dia      4                   DIA                 1.0               16744
2 dia      4                   DIA                 2.0                5763
1 hora     3                   HORA                1.0                4157
30 minuto  2                   MINUTO              30.0               3884
3 dia      4                   DIA                 3.0                3616
4 dia      4                   DIA                 4.0                2357
5 dia      4                   DIA                 5.0                2002
2 hora     3                   HORA                2.0                1870
Name: count, dtype: int64

In [105]:
hist_silver.shape

(311134, 67)

## ✅ IDADE_GESTACIONAL_MOMENTO_REACAO

In [106]:
from utils.not_normalize_idade_gestacional_momento_reacao import normalize_idade_gestacional_momento_reacao

In [107]:
hist_silver = normalize_idade_gestacional_momento_reacao(hist_silver, coluna="IDADE_GESTACIONAL_MOMENTO_REACAO")

In [108]:
hist_silver[['IDADE_GESTACIONAL_MOMENTO_REACAO','IDADE_GESTACIONAL_MOMENTO_REACAO_TIPO_CHAVE','IDADE_GESTACIONAL_MOMENTO_REACAO_TIPO_VALOR','IDADE_GESTACIONAL_MOMENTO_REACAO_VALOR']].value_counts(dropna=False).head(10)

IDADE_GESTACIONAL_MOMENTO_REACAO  IDADE_GESTACIONAL_MOMENTO_REACAO_TIPO_CHAVE  IDADE_GESTACIONAL_MOMENTO_REACAO_TIPO_VALOR  IDADE_GESTACIONAL_MOMENTO_REACAO_VALOR
NaN                               0                                            DESCONHECIDO                                 NaN                                       310802
1 Trimestre                       3                                            TRIMESTRE                                    1.0                                           20
34 Semanas                        1                                            SEMANA                                       34.0                                          19
35 Semanas                        1                                            SEMANA                                       35.0                                          16
9 Semanas                         1                                            SEMANA                                       9.0                  

In [109]:
hist_silver.shape

(311134, 70)

### ✅ CODIGO_ATC

In [123]:
from utils import desagrupar_colunas_pipe

In [124]:
colunas_para_desagrupar = ['ATC_CODE_LEVEL_4']
print(f"Shape antes da desagregação: {hist_silver.shape}") 
hist_silver_dedup = desagrupar_colunas_pipe(hist_silver.rename(columns={'CODIGO_ATC': 'ATC_CODE_LEVEL_4'}), colunas_para_desagrupar)
print(f"Shape depois da desagregação: {hist_silver_dedup.shape}") 

Shape antes da desagregação: (311134, 70)
Shape depois da desagregação: (311134, 70)


In [125]:
hist_silver_dedup["HASH_SILVER"] = build_row_hash(
    hist_silver_dedup, 
    hist_silver_dedup.columns.difference(['ATUALIZADO', 'HASH_BRONZE']).tolist(), 
    algo="sha256", 
    sep="|"
)

In [126]:
hist_silver_dedup.columns

Index(['UF', 'TIPO_ENTRADA_VIGIMED', 'RECEBIDO_DE',
       'IDENTIFICACAO_NOTIFICACAO', 'DATA_INCLUSAO_SISTEMA',
       'DATA_ULTIMA_ATUALIZACAO', 'DATA_NOTIFICACAO', 'TIPO_NOTIFICACAO',
       'NOTIFICACAO_PARENT_CHILD', 'DATA_NASCIMENTO', 'IDADE_MOMENTO_REACAO',
       'GRUPO_IDADE', 'IDADE_GESTACIONAL_MOMENTO_REACAO', 'SEXO', 'GESTANTE',
       'LACTANTE', 'PESO_KG', 'ALTURA_CM', 'REACAO_EVENTO_ADVERSO_MEDDRA',
       'GRAVE', 'GRAVIDADE', 'DESFECHO', 'DATA_INICIO_HORA', 'DATA_FINAL_HORA',
       'DURACAO', 'RELACAO_MEDICAMENTO_EVENTO', 'NOME_MEDICAMENTO_WHODRUG',
       'ACAO_ADOTADA', 'NOTIFICADOR', 'ATUALIZADO', 'HASH_BRONZE', 'UF_CHAVE',
       'UF_VALOR', 'GRAVIDADE_CHAVE', 'GRAVIDADE_VALOR',
       'TIPO_ENTRADA_VIGIMED_VALOR', 'TIPO_ENTRADA_VIGIMED_CHAVE',
       'RECEBIDO_DE_VALOR', 'RECEBIDO_DE_CHAVE', 'TIPO_NOTIFICACAO_VALOR',
       'TIPO_NOTIFICACAO_CHAVE', 'NOTIFICACAO_PARENT_CHILD_VALOR',
       'NOTIFICACAO_PARENT_CHILD_CHAVE', 'GRUPO_IDADE_VALOR',
       'GRUPO_IDADE

In [127]:
hist_silver_dedup.to_parquet(Path(project_root) / "data/02_silver/hist_dim_notificacoes/hist_dim_notificacoes.parquet", index=False)

In [128]:
hist_silver_dedup.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 311134 entries, 0 to 311133
Data columns (total 71 columns):
 #   Column                                       Non-Null Count   Dtype         
---  ------                                       --------------   -----         
 0   UF                                           229379 non-null  object        
 1   TIPO_ENTRADA_VIGIMED                         311064 non-null  object        
 2   RECEBIDO_DE                                  255468 non-null  object        
 3   IDENTIFICACAO_NOTIFICACAO                    311134 non-null  object        
 4   DATA_INCLUSAO_SISTEMA                        311134 non-null  datetime64[ns]
 5   DATA_ULTIMA_ATUALIZACAO                      311064 non-null  datetime64[ns]
 6   DATA_NOTIFICACAO                             158960 non-null  datetime64[ns]
 7   TIPO_NOTIFICACAO                             311134 non-null  object        
 8   NOTIFICACAO_PARENT_CHILD                     1128 non-null    ob

# 🥇 Gold

In [118]:
hist_silver_dedup.columns

Index(['UF', 'TIPO_ENTRADA_VIGIMED', 'RECEBIDO_DE',
       'IDENTIFICACAO_NOTIFICACAO', 'DATA_INCLUSAO_SISTEMA',
       'DATA_ULTIMA_ATUALIZACAO', 'DATA_NOTIFICACAO', 'TIPO_NOTIFICACAO',
       'NOTIFICACAO_PARENT_CHILD', 'DATA_NASCIMENTO', 'IDADE_MOMENTO_REACAO',
       'GRUPO_IDADE', 'IDADE_GESTACIONAL_MOMENTO_REACAO', 'SEXO', 'GESTANTE',
       'LACTANTE', 'PESO_KG', 'ALTURA_CM', 'REACAO_EVENTO_ADVERSO_MEDDRA',
       'GRAVE', 'GRAVIDADE', 'DESFECHO', 'DATA_INICIO_HORA', 'DATA_FINAL_HORA',
       'DURACAO', 'RELACAO_MEDICAMENTO_EVENTO', 'NOME_MEDICAMENTO_WHODRUG',
       'ACAO_ADOTADA', 'NOTIFICADOR', 'ATUALIZADO', 'HASH_BRONZE', 'UF_CHAVE',
       'UF_VALOR', 'GRAVIDADE_CHAVE', 'GRAVIDADE_VALOR',
       'TIPO_ENTRADA_VIGIMED_VALOR', 'TIPO_ENTRADA_VIGIMED_CHAVE',
       'RECEBIDO_DE_VALOR', 'RECEBIDO_DE_CHAVE', 'TIPO_NOTIFICACAO_VALOR',
       'TIPO_NOTIFICACAO_CHAVE', 'NOTIFICACAO_PARENT_CHILD_VALOR',
       'NOTIFICACAO_PARENT_CHILD_CHAVE', 'GRUPO_IDADE_VALOR',
       'GRUPO_IDADE

In [137]:
# Lista base de colunas
gold_cols = [
    'IDENTIFICACAO_NOTIFICACAO',

    #'UF',
    'UF_CHAVE',
    'UF_VALOR',

    #'TIPO_ENTRADA_VIGIMED',
    'TIPO_ENTRADA_VIGIMED_CHAVE',
    'TIPO_ENTRADA_VIGIMED_VALOR',

    #'RECEBIDO_DE',
    'RECEBIDO_DE_CHAVE',
    'RECEBIDO_DE_VALOR',

    'DATA_INCLUSAO_SISTEMA',
    'DATA_ULTIMA_ATUALIZACAO',
    'DATA_NOTIFICACAO',

    #'TIPO_NOTIFICACAO',
    'TIPO_NOTIFICACAO_CHAVE',
    'TIPO_NOTIFICACAO_VALOR',

    #'NOTIFICACAO_PARENT_CHILD',
    'NOTIFICACAO_PARENT_CHILD_CHAVE',
    'NOTIFICACAO_PARENT_CHILD_VALOR',

    'DATA_NASCIMENTO',

    #'IDADE_MOMENTO_REACAO',
    'IDADE_MOMENTO_REACAO_TIPO_CHAVE',
    'IDADE_MOMENTO_REACAO_TIPO_VALOR',
    'IDADE_MOMENTO_REACAO_VALOR',

    #'GRUPO_IDADE',
    'GRUPO_IDADE_CHAVE',
    'GRUPO_IDADE_VALOR',

    #'IDADE_GESTACIONAL_MOMENTO_REACAO',
    'IDADE_GESTACIONAL_MOMENTO_REACAO_TIPO_CHAVE',
    'IDADE_GESTACIONAL_MOMENTO_REACAO_TIPO_VALOR',
    'IDADE_GESTACIONAL_MOMENTO_REACAO_VALOR',

    #'SEXO',
    'SEXO_CHAVE',
    'SEXO_VALOR',

    #'GESTANTE',
    'GESTANTE_CHAVE',
    'GESTANTE_VALOR',

    #'LACTANTE',
    'LACTANTE_CHAVE',
    'LACTANTE_VALOR',

    'PESO_KG',
    'ALTURA_CM',

    'REACAO_EVENTO_ADVERSO_MEDDRA',

    #'GRAVE',
    'GRAVE_CHAVE',
    'GRAVE_VALOR',

    #'GRAVIDADE',
    'GRAVIDADE_CHAVE',
    'GRAVIDADE_VALOR',

    #'DESFECHO',
    'DESFECHO_CHAVE',
    'DESFECHO_VALOR',

    'DATA_INICIO_HORA',
    'DATA_FINAL_HORA',

    #'DURACAO',
    'DURACAO_TIPO_CHAVE',
    'DURACAO_TIPO_VALOR',
    'DURACAO_VALOR',

    #'RELACAO_MEDICAMENTO_EVENTO',
    'RELACAO_MEDICAMENTO_EVENTO_CHAVE',
    'RELACAO_MEDICAMENTO_EVENTO_VALOR',

    'NOME_MEDICAMENTO_WHODRUG',

    #'ACAO_ADOTADA',
    'ACAO_ADOTADA_CHAVE',
    'ACAO_ADOTADA_VALOR',

    #'NOTIFICADOR',
    'NOTIFICADOR_CHAVE',
    'NOTIFICADOR_VALOR',

    'ATUALIZADO', 
    'HASH_BRONZE',
    'HASH_SILVER'
]

# Opcional: aplicar no DataFrame pandas
fat = hist_silver_dedup[gold_cols]

In [138]:
fat["HASH_GOLD"] = build_row_hash(fat, gold_cols, algo="sha256", sep="|")

C:\Users\janet\AppData\Local\Temp\ipykernel_12020\4051699106.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  fat["HASH_GOLD"] = build_row_hash(fat, gold_cols, algo="sha256", sep="|")


In [139]:
fat.to_parquet(Path(project_root) / "data/03_gold/dim_notificacoes/dim_notificacoes.parquet", index=False)

In [140]:
fat.head()

,IDENTIFICACAO_NOTIFICACAO,UF_CHAVE,UF_VALOR,TIPO_ENTRADA_VIGIMED_CHAVE,TIPO_ENTRADA_VIGIMED_VALOR,RECEBIDO_DE_CHAVE,RECEBIDO_DE_VALOR,DATA_INCLUSAO_SISTEMA,DATA_ULTIMA_ATUALIZACAO,DATA_NOTIFICACAO,TIPO_NOTIFICACAO_CHAVE,TIPO_NOTIFICACAO_VALOR,NOTIFICACAO_PARENT_CHILD_CHAVE,NOTIFICACAO_PARENT_CHILD_VALOR,DATA_NASCIMENTO,IDADE_MOMENTO_REACAO_TIPO_CHAVE,IDADE_MOMENTO_REACAO_TIPO_VALOR,IDADE_MOMENTO_REACAO_VALOR,GRUPO_IDADE_CHAVE,GRUPO_IDADE_VALOR,IDADE_GESTACIONAL_MOMENTO_REACAO_TIPO_CHAVE,IDADE_GESTACIONAL_MOMENTO_REACAO_TIPO_VALOR,IDADE_GESTACIONAL_MOMENTO_REACAO_VALOR,SEXO_CHAVE,SEXO_VALOR,GESTANTE_CHAVE,GESTANTE_VALOR,LACTANTE_CHAVE,LACTANTE_VALOR,PESO_KG,ALTURA_CM,REACAO_EVENTO_ADVERSO_MEDDRA,GRAVE_CHAVE,GRAVE_VALOR,GRAVIDADE_CHAVE,GRAVIDADE_VALOR,DESFECHO_CHAVE,DESFECHO_VALOR,DATA_INICIO_HORA,DATA_FINAL_HORA,DURACAO_TIPO_CHAVE,DURACAO_TIPO_VALOR,DURACAO_VALOR,RELACAO_MEDICAMENTO_EVENTO_CHAVE,RELACAO_MEDICAMENTO_EVENTO_VALOR,NOME_MEDICAMENTO_WHODRUG,ACAO_ADOTADA_CHAVE,ACAO_ADOTADA_VALOR,NOTIFICADOR_CHAVE,NOTIFICADOR_VALOR,ATUALIZADO,HASH_BRONZE,HASH_SILVER,HASH_GOLD
0,BR-ANVISA-300212656,25,SP,2,EMPRESAS FARMACEUTICAS,5,EMPRESA FARMACEUTICA,2023-09-28,2023-09-28,NaT,1,NOTIFICACAO ESPONTANEA,0,DESCONHECIDO,1990-01-31,1,ANO,30.0,0,DESCONHECIDO,0,DESCONHECIDO,NaN,1,FEMININO,2,NĂO,2,NĂO,68.0,165,Hemiparesia,1,SIM,1,OUTRO EFEITO CLINICAMENTE SIGNIFICATIVO,1,RECUPERADO,2021-01-25,2021-02-06,4,DIA,12.0,2,CONCOMITANTE,Tamisa,2,NAO APLICAVEL,4,MEDICO,2026-01-15,c540f6f2873cded948f9a6156d6a92cf9f59b7e110f84ebc4a988ccdb13b1c3d,3c04663c8c6a69bba2510ab19fa043d1d6670b81e428579845379ee75e9559fa,95ef1ec6b9f7b04d5d423f561ef99f6fe7ea4ce5f916a9c2154c35ecfe8f6692
1,BR-ANVISA-300208322,25,SP,2,EMPRESAS FARMACEUTICAS,5,EMPRESA FARMACEUTICA,2023-09-01,2023-09-01,NaT,1,NOTIFICACAO ESPONTANEA,0,DESCONHECIDO,NaT,0,DESCONHECIDO,NaN,0,DESCONHECIDO,0,DESCONHECIDO,NaN,1,FEMININO,2,NĂO,2,NĂO,None,None,Cefaleia,2,NĂO,1,OUTRO EFEITO CLINICAMENTE SIGNIFICATIVO,0,DESCONHECIDO,2021-01-22,NaT,0,DESCONHECIDO,NaN,1,SUSPEITO,CoronaVac,2,NAO APLICAVEL,2,CONSUMIDOR OU OUTRO NAO PROFISSIONAL DE SAUDE,2026-01-15,1f6805d588f843fe6f27dc43ae2026701e8d0400f3a10108c262015650b0b5b5,716905c5a2cc460902e5682c94bd1379f674208b0c6ef526b96e2a8a882800c7,8db30198a3122b4ee42039e05a0d958d0caa1efd8f2ece8b4f4ad28b6140f12e
2,BR-ANVISA-300214015,25,SP,2,EMPRESAS FARMACEUTICAS,5,EMPRESA FARMACEUTICA,2023-10-06,2023-10-06,NaT,1,NOTIFICACAO ESPONTANEA,0,DESCONHECIDO,1971-05-22,1,ANO,49.0,0,DESCONHECIDO,0,DESCONHECIDO,NaN,2,MASCULINO,2,NĂO,2,NĂO,None,None,Nefrolitíase,1,SIM,2,HOSPITALIZACAO,0,DESCONHECIDO,2021-02-03,NaT,0,DESCONHECIDO,NaN,2,CONCOMITANTE,Metformina,0,DESCONHECIDO,4,MEDICO,2026-01-15,0bf848a27d155b0a8bdebe8473dfb8c83b42d7d1e9123a666eceea3d811df4f9,6c2d20e95e641e916296e029f52f032403b56ca7c100f0eb57e8b9a515ffd808,c37b4a135723f33dda015d5cc9c8ad5141c309de6b165394c41b6e4a6910c648
3,BR-ANVISA-300345102,25,SP,6,"EREPORTING - INDUSTRIA, CARGA E2B",5,EMPRESA FARMACEUTICA,2025-07-21,2025-07-21,NaT,1,NOTIFICACAO ESPONTANEA,0,DESCONHECIDO,1974-11-23,1,ANO,46.0,0,DESCONHECIDO,0,DESCONHECIDO,NaN,2,MASCULINO,2,NĂO,2,NĂO,None,None,Choque anafilático,1,SIM,3,AMEACA A VIDA,0,DESCONHECIDO,2021-03-02,NaT,0,DESCONHECIDO,NaN,1,SUSPEITO,Soro antibotropico (pentavalente),0,DESCONHECIDO,3,FARMACEUTICO,2026-01-15,14d6e7f81906f9def902e0ce10e10c8c1c8139bda38a39a700309ff9bd00dc03,9f6fd05dec6049f2a833cf58768c4b9c8aba966095ad08afcd5132ec4ac167ae,0924e8acbbdb0377c67deb7745391f6871768235ce09f7a1c6521b859708a681
4,BR-ANVISA-300212385,25,SP,2,EMPRESAS FARMACEUTICAS,5,EMPRESA FARMACEUTICA,2023-09-27,2023-09-27,NaT,1,NOTIFICACAO ESPONTANEA,0,DESCONHECIDO,NaT,0,DESCONHECIDO,NaN,0,DESCONHECIDO,0,DESCONHECIDO,NaN,2,MASCULINO,2,NĂO,2,NĂO,None,None,Abscesso nasal,1,SIM,1,OUTRO EFEITO CLINICAMENTE SIGNIFICATIVO,0,DESCONHECIDO,NaT,NaT,0,DESCONHECIDO,NaN,1,SUSPEITO,Vacina adsorvida COVID-19 (inativada),0,DESCONHECIDO,2,CONSUMIDOR OU OUTRO NAO PROFISSIONAL DE SAUDE,2026-01-15,76a79c5954923381ce6ba55291fc63cc158e622993be2b